# 1T Classifier Signals to ThetaData Options Backtest

This notebook tests the equity classifier as a source of option-tradable entry/exit windows.

The pipeline is intentionally shared with the oracle-options notebook:

- Build actual classifier signal windows only; daily non-events are not included.
- Retrieve viable ThetaData options at the entry date.
- Label option candidates from realized option returns and mean-variance basket weights for supervised selector research.
- Simulate selected option trades with Optopsy, a third-party options backtesting engine.

The option rankers use option-chain features by default. Realized holding period, realized underlying return, and equity signal metadata are carried as diagnostics, not default model inputs.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
if REPO_ROOT.name != 'quant-orchestrator':
    REPO_ROOT = next(parent for parent in Path.cwd().resolve().parents if parent.name == 'quant-orchestrator')
WAREHOUSE_ROOT = REPO_ROOT.parent / 'quant-warehouse'
for path in (REPO_ROOT, WAREHOUSE_ROOT):
    resolved = str(path.resolve())
    if resolved in sys.path:
        sys.path.remove(resolved)
    sys.path.insert(0, resolved)
for module_name in list(sys.modules):
    if module_name == 'quant_orchestrator' or module_name.startswith('quant_orchestrator.'):
        del sys.modules[module_name]
    if module_name == 'quant_warehouse' or module_name.startswith('quant_warehouse.'):
        del sys.modules[module_name]

from quant_orchestrator.research_tools import (
    OptopsyExecutionConfig,
    OptionMvBasketConfig,
    OptionRetrievalConfig,
    OracleOptionExperimentConfig,
    SharedSplitConfig,
    build_classifier_signal_trade_windows,
    run_trade_window_option_experiment,
)

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)
print('repo_root', REPO_ROOT)
print('warehouse_root', WAREHOUSE_ROOT)

repo_root /home/jlee153232/PycharmProjects/quant-orchestrator
warehouse_root /home/jlee153232/PycharmProjects/quant-warehouse


In [2]:
SCORE_PATH = REPO_ROOT / 'artifacts/orchestrator/files/ml_trading_gpu_rf_shared_book_1t_dagster_smoke/run_f6b98a7b10ab4c7cb26ef41ece402750/strategy_scores.csv'
AE_SCORE_PATH = REPO_ROOT / 'artifacts/orchestrator/files/ml_trading_gpu_rf_autoencoder_shared_book_1t_dagster_smoke/run_2e809e7e7bab47a6920813736b0f7c4b/strategy_scores.csv'

SYMBOLS = ('AAPL', 'AMZN', 'AVGO', 'GOOG', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA')
PRICE_START = '2018-01-01'
PRICE_END = '2026-06-24'
ENTRY_THRESHOLD = 0.50
EXIT_THRESHOLD = 0.50
TOP_K = 5

scores = pd.read_csv(SCORE_PATH)
display(scores.head())
print('score_rows', len(scores), 'symbols', scores['symbol'].nunique(), 'strategy_sources', scores['strategy_source'].nunique())

,strategy_source,source,family,symbol,date,long_score,short_score,long_exit_score,short_exit_score,classifier_long_score,classifier_short_score,ae_familiarity,ae_recon_error,ae_latent_distance,net_score,model_count
0,ensemble_mean,ensemble,mean,AAPL,2020-01-02,0.772135,0.227865,0.772135,0.227865,0.772135,0.227865,1.0,0.0,0.0,0.544270,15
1,ensemble_mean,ensemble,mean,AAPL,2020-01-03,0.786879,0.213121,0.786879,0.213121,0.786879,0.213121,1.0,0.0,0.0,0.573758,15
2,ensemble_mean,ensemble,mean,AAPL,2020-01-06,0.773988,0.226012,0.773988,0.226012,0.773988,0.226012,1.0,0.0,0.0,0.547976,15
3,ensemble_mean,ensemble,mean,AAPL,2020-01-07,0.776897,0.223103,0.776897,0.223103,0.776897,0.223103,1.0,0.0,0.0,0.553793,15
4,ensemble_mean,ensemble,mean,AAPL,2020-01-08,0.766436,0.233564,0.766436,0.233564,0.766436,0.233564,1.0,0.0,0.0,0.532872,15


score_rows 338416 symbols 13 strategy_sources 16


## Build Classifier Trade Windows

The classifier scores are collapsed into actual held trade windows. This is the same event-only rule used for target/event datasets: if there is no entry and exit event, there is no training row for the option selector.

In [3]:
trade_windows = build_classifier_signal_trade_windows(
    scores,
    strategy_sources=('ensemble_mean',),
    variant='long_short',
    top_k=TOP_K,
    entry_threshold=ENTRY_THRESHOLD,
    exit_threshold=EXIT_THRESHOLD,
)
display(trade_windows)
print('trade_windows', len(trade_windows), 'symbols', trade_windows['symbol'].nunique() if not trade_windows.empty else 0)

,symbol,side,entry_date,exit_date,strategy_source,source,family,source_family,variant,top_k,equity_signal_score,equity_exit_signal_score,planned_holding_days,ae_familiarity,ae_recon_error,ae_latent_distance,trade_id
0,AAPL,long,2020-01-02,2024-09-03,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.772135,0.500623,1706,1.0,0.0,0.0,ensemble_mean|long_short|AAPL|2020-01-02|long
1,AMZN,long,2020-01-02,2026-06-24,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.762867,NaN,2365,1.0,0.0,0.0,ensemble_mean|long_short|AMZN|2020-01-02|long
2,AVGO,short,2020-01-02,2020-03-18,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.769141,0.624394,76,1.0,0.0,0.0,ensemble_mean|long_short|AVGO|2020-01-02|short
3,BRK-A,long,2020-01-02,2026-06-24,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.761963,NaN,2365,1.0,0.0,0.0,ensemble_mean|long_short|BRK-A|2020-01-02|long
4,BRK-B,long,2020-01-02,2026-06-24,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.758827,NaN,2365,1.0,0.0,0.0,ensemble_mean|long_short|BRK-B|2020-01-02|long
5,GOOG,long,2020-03-18,2025-10-28,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.644366,0.500458,2050,1.0,0.0,0.0,ensemble_mean|long_short|GOOG|2020-03-18|long
6,LLY,short,2024-09-03,2024-11-15,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.566217,0.502065,73,1.0,0.0,0.0,ensemble_mean|long_short|LLY|2024-09-03|short
7,TSLA,short,2024-11-15,2025-03-10,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.570757,0.500783,115,1.0,0.0,0.0,ensemble_mean|long_short|TSLA|2024-11-15|short
8,MSFT,long,2025-03-10,2026-06-24,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.574462,NaN,471,1.0,0.0,0.0,ensemble_mean|long_short|MSFT|2025-03-10|long
9,AVGO,short,2025-10-28,2026-03-27,ensemble_mean,ensemble,mean,ensemble.mean,long_short,5,0.572326,0.501817,150,1.0,0.0,0.0,ensemble_mean|long_short|AVGO|2025-10-28|short


trade_windows 11 symbols 10


## AE Familiarity Hook

The AE hook is an entry filter on classifier-generated trade windows. It is not used as a default option-ranker feature here; this keeps the option experiment focused on option-chain features and the selected equity signal windows.

In [4]:
if AE_SCORE_PATH.exists():
    ae_scores = pd.read_csv(AE_SCORE_PATH)
    ensemble_ae = ae_scores.loc[ae_scores['strategy_source'].eq('ensemble_mean')]
    thresholds = [None, float(ensemble_ae['ae_familiarity'].quantile(0.75)), 0.95, 0.98]
    ae_rows = []
    for threshold in thresholds:
        windows = build_classifier_signal_trade_windows(
            ae_scores,
            strategy_sources=('ensemble_mean',),
            variant='long_short',
            top_k=TOP_K,
            entry_threshold=ENTRY_THRESHOLD,
            exit_threshold=EXIT_THRESHOLD,
            min_ae_familiarity=threshold,
        )
        ae_rows.append({'min_ae_familiarity': threshold, 'trade_windows': len(windows), 'symbols': windows['symbol'].nunique() if not windows.empty else 0})
    display(pd.DataFrame(ae_rows))
else:
    print('AE score artifact not found:', AE_SCORE_PATH)

,min_ae_familiarity,trade_windows,symbols
0,NaN,5,5
1,0.42073,5,5
2,0.95000,5,5
3,0.98000,7,5


## Run Option Retrieval, Ranking, MV Basket Selection, and Optopsy

In [5]:
config = OracleOptionExperimentConfig(
    experiment_name='classifier_1t_option_signal_windows',
    symbols=tuple(sorted(scores['symbol'].dropna().astype(str).str.upper().unique())),
    price_start=PRICE_START,
    price_end=PRICE_END,
    split=SharedSplitConfig(insample_end='2024-12-31'),
    retrieval=OptionRetrievalConfig(max_candidates_per_trade=40),
    execution=OptopsyExecutionConfig(capital=100_000.0, quantity=1, max_positions=5, multiplier=100, selector='first'),
    mv_basket=OptionMvBasketConfig(enabled=True, max_legs=4, min_predicted_weight=0.02),
    artifact_dir=str(REPO_ROOT / 'artifacts/options/classifier_1t_option_signal_windows/latest'),
    log_mlflow=True,
)
result = run_trade_window_option_experiment(config, trade_windows)
print('elapsed_seconds', round(result.elapsed_seconds, 2))
print('mlflow_run_id', result.mlflow_run_id)
print('trade_windows', len(result.oracle_trades), 'option_rows', len(result.option_panel), 'train_rows', len(result.train_panel), 'eval_rows', len(result.eval_panel))

/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value enco

elapsed_seconds 3.13
mlflow_run_id 4085db2cdeab44f19e4815c84678db81
trade_windows 8 option_rows 155 train_rows 40 eval_rows 115


In [6]:
display(result.selector_summary)
display(result.optopsy_summary)
display(result.symbol_summary)
Markdown(result.analysis_markdown)

,selector,trades,symbols,mean_return,median_return,win_rate,avg_spread_pct,avg_abs_moneyness,avg_dte,legs,avg_legs_per_trade
0,model_ranker,3,3.0,2.149260,1.657005,1.000000,0.022204,0.056221,51.333333,NaN,NaN
1,oracle_best_possible,3,3.0,2.149260,1.657005,1.000000,0.022204,0.056221,51.333333,NaN,NaN
2,fixed_near_atm,3,3.0,0.172426,0.158879,0.666667,0.019479,0.001925,46.333333,NaN,NaN
3,highest_liquidity,3,3.0,0.085333,-0.101466,0.333333,0.015708,0.015001,46.333333,NaN,NaN
4,lowest_spread,3,3.0,0.237648,0.355482,0.666667,0.012792,0.021599,46.333333,NaN,NaN
5,model_mv_basket,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,oracle_mv_basket,3,3.0,1.450640,1.370158,1.000000,0.019769,0.034407,58.333333,12.0,4.0


,selector,framework,selected_rows,closed_trades,equity_points,final_equity,total_trades,winning_trades,losing_trades,win_rate,total_pnl,total_return,avg_pnl,avg_win,avg_loss,max_win,max_loss,profit_factor,max_drawdown,avg_days_in_trade,sharpe_ratio,sortino_ratio,var_95,cvar_95,calmar_ratio,omega_ratio,tail_ratio,note
0,model_ranker,optopsy,3,3,3.0,108829.500000,3.0,3.0,0.0,1.000000,8829.500000,0.088295,2943.166667,2943.166667,0.00,4287.500000,1337.500000,inf,0.00000,51.333333,1.249052,0.000000,0.485634,0.355482,0.000000,0.000000,0.0,NaN
1,oracle_best_possible,optopsy,3,3,3.0,108829.500000,3.0,3.0,0.0,1.000000,8829.500000,0.088295,2943.166667,2943.166667,0.00,4287.500000,1337.500000,inf,0.00000,51.333333,1.249052,0.000000,0.485634,0.355482,0.000000,0.000000,0.0,NaN
2,fixed_near_atm,optopsy,3,3,3.0,103962.000000,3.0,2.0,1.0,0.666667,3962.000000,0.039620,1320.666667,2781.500000,-1601.00,5138.000000,-1601.000000,3.474703,0.00000,46.333333,0.564384,1.924782,-0.883270,-0.999064,1.435239,3.517212,0.0,NaN
3,highest_liquidity,optopsy,3,3,3.0,103590.500000,3.0,1.0,2.0,0.333333,3590.500000,0.035905,1196.833333,5138.000000,-773.75,5138.000000,-1322.500000,3.320194,-0.00228,46.333333,0.520208,2.090082,-0.910147,-1.000000,1.346640,3.365823,0.0,NaN
4,lowest_spread,optopsy,3,3,3.0,104580.500000,3.0,2.0,1.0,0.666667,4580.500000,0.045805,1526.833333,3237.750000,-1895.00,5138.000000,-1895.000000,3.417150,0.00000,46.333333,0.624880,1.870317,-0.864452,-1.000000,1.400116,3.445984,0.0,NaN
5,model_mv_basket,optopsy,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no selected trades
6,oracle_mv_basket,optopsy,12,3,3.0,110317.938848,3.0,3.0,0.0,1.000000,10317.938848,0.103179,3439.312949,3439.312949,0.00,5031.256731,456.264085,inf,0.00000,56.000000,1.129574,0.000000,0.282553,0.161921,0.000000,0.000000,0.0,NaN


,symbol,side,trades,mean_return,win_rate
2,MSFT,long,1,4.435294,1.0
1,META,long,1,1.657005,1.0
0,AVGO,short,1,0.355482,1.0


## Written Analysis

- Oracle equity trades generated: 8.
- Shared split: in-sample <= 2024-12-31, out-of-sample >= 2025-01-01.
- Option retrieval produced 155 contract rows across 4 oracle trade windows.
- Train rows: 40; eval rows: 115.
- ThetaData option feature columns available in this run: liquidity_score.
- Greeks/IV missing from the cached chains for this run: delta, gamma, theta, vega, rho, iv. The feature-engineering API will include them automatically when present in ThetaData storage.
- model_ranker: 3 eval trades, mean=214.93%, median=165.70%, win_rate=100.00%.
- oracle_best_possible: 3 eval trades, mean=214.93%, median=165.70%, win_rate=100.00%.
- fixed_near_atm: 3 eval trades, mean=17.24%, median=15.89%, win_rate=66.67%.
- highest_liquidity: 3 eval trades, mean=8.53%, median=-10.15%, win_rate=33.33%.
- lowest_spread: 3 eval trades, mean=23.76%, median=35.55%, win_rate=66.67%.
- model_mv_basket: no eval trades.
- oracle_mv_basket: 3 eval trades, mean=145.06%, median=137.02%, win_rate=100.00%.
- Portfolio execution is simulated by the third-party Optopsy engine; Quant Orchestrator only adapts selected contracts into Optopsy's raw-trade schema.
- Optopsy model_ranker: closed_trades=3, final_equity=108,829.50, total_return=8.83%.
- Optopsy oracle_best_possible: closed_trades=3, final_equity=108,829.50, total_return=8.83%.
- Optopsy fixed_near_atm: closed_trades=3, final_equity=103,962.00, total_return=3.96%.
- Optopsy highest_liquidity: closed_trades=3, final_equity=103,590.50, total_return=3.59%.
- Optopsy lowest_spread: closed_trades=3, final_equity=104,580.50, total_return=4.58%.
- Optopsy model_mv_basket: closed_trades=0, final_equity=nan, total_return=nan%.
- Optopsy oracle_mv_basket: closed_trades=3, final_equity=110,317.94, total_return=10.32%.
- These artifacts are experiment products, not permanent Quant Warehouse source-of-truth tables.

## Written Analysis

The first 1T smoke run produced very few actual ensemble classifier windows. That is a useful finding: with `top_k=5`, `entry_threshold=0.50`, and signal-based exits, the ensemble mostly holds long positions for years rather than producing many option-sized rotation events.

Because the option panel is small, the single-leg option ranker can run, but the learned MV basket selector has limited training support. The oracle MV basket remains useful as an upper-bound diagnostic for whether multi-leg baskets could add value once the classifier creates enough trade windows.

The next useful research step is not to add more option modeling complexity. It is to generate more realistic classifier trade windows for options: shorter planned horizons, option-liquidity-aware universes, and feature-family-specific signals rather than only the ensemble mean.